# Advanced RAG Techniques
- **Hybrid Search**: Combine keyword search (BM25) with semantic search (FAISS) to get the best of both worlds using the `EnsembleRetriever`.
- **Multi-vector RAG**: Handle complex documents by creating different representations (summaries and raw chunks) for smarter retrieval with the `MultiVectorRetriever`.



## 0. Setup

In [ ]:
!pip install langchain langchain-google-genai langchain-community faiss-cpu rank_bm25

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path='../../.env')  # Specify the path to your .env file

# Access the environment variable
api_key = os.getenv('OPENAI_API_KEY')

# Check if the variable is loaded
if api_key or api_key == "":
    print(f"API key loaded successfully.")
else:
    print("Failed to load API key.")

from openai import OpenAI
client = OpenAI(api_key=api_key)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.schema import Document

# Main components
llm = ChatOpenAI(
    model= "gpt-4o-mini",
    openai_api_key=api_key,
    temperature=0.3,
)
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002",
    api_key=api_key
)

# Sample documents for testing
docs = [
    Document(
        page_content="The HTTP 404 Not Found error occurs when the server cannot find the requested resource. This can be caused by a mistyped URL or a broken link.",
        metadata={"source": "doc_http_404"}
    ),
    Document(
        page_content="SSH, or Secure Shell, is a protocol that allows secure remote access to servers. To connect, use the default port 22 and an SSH client.",
        metadata={"source": "doc_ssh_remote_access"}
    ),
    Document(
        page_content="To view running containers in Docker, use the 'docker ps' command. If the error 'Cannot connect to the Docker daemon' appears, check if the Docker service is active.",
        metadata={"source": "doc_docker_commands"}
    ),
    Document(
        page_content="The corporate vacation policy guarantees 30 days of rest per year. Employees must access the internal HR portal and fill out the 'FRM-01-VACATION' form to formalize the request.",
        metadata={"source": "doc_vacation_with_form"}
    ),
    Document(
        page_content="To request vacation, employees must access the HR system and follow the steps described in the manual, filling out the correct form for approval.",
        metadata={"source": "doc_vacation_without_form_name"}
    )
]

## 1. Hybrid Search with `EnsembleRetriever`

Vector search is excellent for semantics but poor for keywords. Keyword search (BM25) is the opposite. Hybrid search combines the two. LangChain's `EnsembleRetriever` does this elegantly.

**Scenario**: The user searches for the exact term "FRM-01-FERIAS". A purely vector search might not give due weight to this specific code.

In [ ]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.vectorstores import FAISS

# Keyword Retriever (Sparse)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2

# Retriever
faiss_vectorstore = FAISS.from_documents(docs, embeddings)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

# Ensemble Retriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5]
)

In [ ]:
# --- Retrieval Test ----------------------------------------------------------
query_keyword = "How do I request vacation using the FRM-01-FERIAS form?"

def show_results(title, docs, term="FRM-01-FERIAS"):
    print(f"\n--- {title} ---")
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "no_source")
        print(f"    {d.page_content[:120]}...")  # Uncomment to see the beginning of the text

print(f"--- Searching for: '{query_keyword}' ---")

# Vector Search (FAISS)
docs_faiss = faiss_retriever.invoke(query_keyword)
show_results("Vector Search Results (FAISS)", docs_faiss)

# Hybrid Search (BM25 + FAISS via EnsembleRetriever)
docs_ensemble = ensemble_retriever.invoke(query_keyword)
show_results("Hybrid Search Results (EnsembleRetriever)", docs_ensemble)

# Simple automatic analysis
faiss_top = docs_faiss[0].metadata.get("source")
ensemble_top = docs_ensemble[0].metadata.get("source")



## 2. Multi-vector RAG with `MultiVectorRetriever`

For long or complex documents, embedding small chunks can cause the RAG to lose the overall context. Multi-vector RAG solves this by first creating and searching summaries of the documents, and then retrieving the raw chunks for response generation.

**Scenario**: We have a long document and want the initial search to consider the overall context of the document, not just small excerpts.

In [ ]:
import uuid
from langchain.storage import InMemoryStore
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema import Document

# Long document for the example
doc_long = [
    Document(
        page_content="""
    Introduction to Cybersecurity (2024)...
    ...
    One of the most common attack techniques is Phishing...
    ...
    Conclusion: Stay updated... Two-factor authentication (2FA) should be mandatory.
    """,
        metadata={"source": "cybersecurity_guide.pdf", "year": 2024}
    )
]

# 1. Splitter to divide the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300)
doc_chunks = text_splitter.split_documents(doc_long)

# 2. Summary Generator
def generate_summaries(docs, llm_model):
    """Generates summaries for a list of documents."""
    prompt = ChatPromptTemplate.from_template(
        "Summarize the following document in one sentence: {document}"
    )
    chain = prompt | llm_model
    summaries = chain.batch([{"document": doc.page_content} for doc in docs])
    return [s.content for s in summaries]



In [ ]:
## Configuring the MultiVectorRetriever

doc_ids = [str(uuid.uuid4()) for _ in doc_chunks]

summary_chunks = generate_summaries(doc_chunks, llm)

store = InMemoryStore()
store.mset(list(zip(doc_ids, doc_chunks)))

# Create a vector store for the summaries, loading the origin ID from metadata

summary_vectorstore = FAISS.from_texts(
    summary_chunks,
    embeddings,
    metadatas=[{"doc_id": doc_ids[i]} for i in range(len(summary_chunks))]
)

# Retriever that searches the vector store and returns the complete chunk via the store

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=summary_vectorstore,
    docstore=store,
    id_key="doc_id",
    search_kwargs={'k': 1}
)



In [ ]:
# --- Retrieval Test ---
query_summary = "what is the main defense against cyber attacks?"

retrieved_docs = multi_vector_retriever.invoke(query_summary)

print(f"--- Searching for: '{query_summary}' ---\n")
print("--- Original Document Retrieved via Summary (MultiVectorRetriever) ---")
if retrieved_docs:
    print(retrieved_docs[0].page_content)
else:
    print("No documents were retrieved.")

print("\n💡 **Analysis**: The search was conducted on the summaries, which capture the essence of each part of the document. "
      "By finding the relevant summary about 'defenses', the retriever provided the detailed original chunk containing "
      "the precise answer.")